In [5]:
%pip install -q torch nvidia-ml-py3 librosa numpy

# Using CUDA 13.0
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
# pip install transformers

import os
import gc
import json
import numpy as np
import pickle
import gzip
import librosa # For audio processing
from pathlib import Path
from typing import Tuple, List, Dict
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, DataLoader, random_split
from torch.amp import GradScaler, autocast
from transformers import ASTModel, ASTConfig, ASTFeatureExtractor

import pynvml
import psutil
from tqdm import tqdm
from sklearn.metrics import f1_score

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

class Config:
    # Paths
    PARENT_DIR = os.path.dirname(os.getcwd())
    DATA_DIR = PARENT_DIR + "/Data"
    PREPROCESSED_DIR = PARENT_DIR + "/Preprocessed"
    OUTPUT_DIR = PARENT_DIR + "/Model_Output"

    # Preprocessor parameters
    AUDIO_EPOCH_DURATION=10 # 10s clips preferred for AST, 30s causes memory errors
    AUDIO_SAMPLE_RATE=16000
    NUM_AUDIO_FILES=50

    # Feature parameters
    USE_PREEXTRACTED_FEATURES = True
    
    # Model parameters
    CONTEXT_EPOCHS = 14
    OUTPUT_EPOCHS = 10
    NUM_CLASSES = 3
    DROPOUT = 0.3
    LSTM_ENABLED = True
    
    # Training parameters
    BATCH_SIZE = 128
    GRADIENT_ACCUMULATION_STEPS = 1 # Effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    PATIENCE = 10
    VAL_SPLIT = 0.2
    
    # DataLoader parameters
    NUM_WORKERS = 8 #MAX 
    CACHE_SIZE = 48 # 2 Bad Folders
    USE_COMPRESSION = True
    
    # Memory optimization
    USE_MIXED_PRECISION = True
    
    # Class weights
    CLASS_WEIGHTS = [1.0, 1.3, 2.1]

config = Config()

Note: you may need to restart the kernel to use updated packages.


In [2]:
class AudoPreprocessor:
    # 30s audio clips at 16kHz by default per study (see below for actual length)
    def __init__(self, data_dir, output_dir, epoch_duration = 30, sample_rate = 16000, use_compression=False, extract_features=True):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.epoch_duration = epoch_duration
        self.sample_rate = sample_rate
        self.use_compression = use_compression
        self.extract_features = extract_features
        
        # AST Input Shape
        self.target_length = sample_rate * epoch_duration

        # Extract Features
        if self.extract_features:
            self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            
            # Load model for feature extraction
            self.ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            self.ast_model.eval()
            
            # Move to GPU if available
            if torch.cuda.is_available():
                self.ast_model = self.ast_model.cuda()
                print("    Feature extraction will use GPU")
        
        # Make output directory if it doesn't exist
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\nInitialized AudoPreprocessor")
        print(f"    Data_dir: {data_dir}")
        print(f"    Output_dir: {output_dir}")
        print(f"    Epoch Duration: {epoch_duration}s")
        print(f"    Sample Rate: {sample_rate}Hz")
        print(f"    Samples per epoch: {self.target_length}")
        print(f"    Compression: {'Enabled (gzip)' if use_compression else 'Disabled'}")

    def extract_features_batch(self, audio_epochs):
        if not self.extract_features:
            return audio_epochs

        # Batch Processing
        batch_size = 32
        all_features = []

        with torch.no_grad():
            for i in range(0, len(audio_epochs), batch_size):
                batch = audio_epochs[i:i+batch_size]
                
                # Feature extraction
                inputs = self.ast_feature_extractor(
                    list(batch),
                    sampling_rate=self.sample_rate,
                    return_tensors="pt"
                )
                
                # Move to GPU if available
                if torch.cuda.is_available():
                    inputs = {k: v.cuda() for k, v in inputs.items()}
                
                # Extract features
                outputs = self.ast_model(**inputs)
                features = outputs.last_hidden_state[:, 0, :]  # CLS token
                
                all_features.append(features.cpu().numpy())
        
        return np.vstack(all_features)
    
    def load_annotations(self, folder_id) -> Dict:
        # load annotations from json like 01_annotation.json
        annotation_path = self.data_dir / folder_id / f"{folder_id}_annotation.json"
        
        if (not annotation_path.exists()):
            raise FileNotFoundError(f"Annotation file not found: {annotation_path}")
        
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)
            
        # Debug print statements
        print(f"\nLoaded annotations from {annotation_path}")
        print(f"    Record Start: {annotations['record_start']}s")
        print(f"    Awake Intervals: {len(annotations['awake_intervals'])}")
        print(f"    Events: {len(annotations['events'])}")
        
        return annotations
    
    def load_audio(self, folder_id) -> Tuple[np.ndarray, int]:
        # load audio file like 01_phone.wav
        audio_path = self.data_dir / folder_id / f"{folder_id}_phone.wav"
        
        if (not audio_path.exists()):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        
        # Librosa audio loading
        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        
        # Debug print statements
        print(f"\nLoaded audio from {audio_path}")
        print(f"    Audio Shape: {audio.shape}")
        print(f"    Sample Rate: {sr}Hz")
        print(f"    Duration: {len(audio)/sr:.2f}s")
        
        return audio, sr
    
    # Check if time point is within any awake interval
    def is_awake(self, time_point: float, awake_intervals: List[Tuple[float]]) -> bool:
        for start, end in awake_intervals:
            if start <= time_point <= end:
                return True
        return False
        
    # Extract epoch labels based on annotations
    def extract_epoch_labels(self, epoch_start: float, epoch_end: float, events: List[Dict], awake_intervals: List[List[float]]) -> int:
        # Check if epoch during awake interval
        if (self.is_awake(epoch_start, awake_intervals) or self.is_awake(epoch_end, awake_intervals)):
            return -1  # Awake
        
        label = 0 # Default to no event (1 = osa [obstructive sleep apnea], 2 = hyp [hypnopnea])
        
        # Gonna prioritize in order hypo > osa > none
        for event in events:
            event_start = event['evnet_start']  # Note: typo in original data
            event_end = event_start + event['event_duration']
            event_type = event['event_type']
            
            # Check if event overlaps with epoch
            if not (event_end < epoch_start or event_start > epoch_end):
                if event_type == 'hypo':
                    label = max(label, 2)
                elif event_type == 'osa':
                    label = max(label, 1)
        
        return label
    
    # Create epochs from audio data and label them
    def create_epochs(self, folder_id: str) -> Tuple[np.ndarray, np.ndarray]:
        # Load data
        annotations = self.load_annotations(folder_id)
        audio, sr = self.load_audio(folder_id)
        
        # Extract Annotations
        record_start = annotations['record_start']
        awake_intervals = annotations['awake_intervals']
        events = annotations['events']
        
        # Number of epochs
        audio_duration = len(audio) / sr
        num_epochs = int(np.floor(audio_duration / self.epoch_duration))
        
        print(f"\nCreating {num_epochs} epochs of {self.epoch_duration}s each from audio of duration {audio_duration:.2f}s")
        
        # Pre-allocate arrays for better memory efficiency
        max_possible_epochs = num_epochs
        epoch_array = []
        label_array = []
        
        label_counts = {-1: 0, 0: 0, 1: 0, 2: 0}
        
        for i in range(num_epochs):
            epoch_start_sample = i * self.target_length
            epoch_end_sample = (i + 1) * self.target_length
            
            # Handle last epoch case if too short
            if epoch_end_sample > len(audio):
                break
            
            epoch_audio = audio[epoch_start_sample:epoch_end_sample]
            
            # Actual start and end time per recording start
            epoch_start_time = record_start + (i * self.epoch_duration)
            epoch_end_time = epoch_start_time + self.epoch_duration
            
            label = self.extract_epoch_labels(epoch_start_time, epoch_end_time, events, awake_intervals)
            
            # Don't care if awake
            if label == -1:
                label_counts[-1] += 1
                continue
            
            epoch_array.append(epoch_audio)
            label_array.append(label)
            label_counts[label] += 1
        
        epoch_array = np.array(epoch_array, dtype=np.float32)
        label_array = np.array(label_array, dtype=np.int32)
        
        # Extract features if enabled
        if self.extract_features:
            print(f"    Extracting AST features...")
            epoch_array = self.extract_features_batch(epoch_array)
            print(f"    Feature shape: {epoch_array.shape}")
        
        print(f"\nEpoch Statistics for folder {folder_id}:")
        print(f"    Total Epochs: {num_epochs}")
        print(f"    Processed Epochs Saved: {len(label_array)}")
        print(f"    Awake Epochs Skipped: {label_counts[-1]}")
        print(f"    No Event Epochs: {label_counts[0]}")
        print(f"    OSA Event Epochs: {label_counts[1]}")
        print(f"    Hypopnea Event Epochs: {label_counts[2]}")
        
        # Clear audio from memory
        del audio
        gc.collect()
        
        return epoch_array, label_array
    
    def save_folder_individually(self, folder_id, epochs, labels):
        file_ext = ".pkl.gz" if self.use_compression else ".pkl"
        folder_output_path = self.output_dir / f"folder_{folder_id}{file_ext}"
        
        data = {
            'epochs': epochs,
            'labels': labels,
            'folder_id': folder_id,
            'sample_rate': self.sample_rate,
            'epoch_duration': self.epoch_duration
        }
        
        # Use highest protocol for better compression and speed
        if self.use_compression:
            with gzip.open(folder_output_path, 'wb', compresslevel=6) as f:
                pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            with open(folder_output_path, 'wb') as f:
                pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        
        file_size_mb = folder_output_path.stat().st_size / (1024 * 1024)
        print(f"    Saved to {folder_output_path} ({file_size_mb:.2f} MB)")
        
        return folder_output_path
    
    def process_folders(self, folder_ids: List[str] = None):
        # If no folder IDs provided, process all folders in data_dir
        if (not folder_ids):
            print ("\nNo folder IDs provided. Defaulting to folders 01-50.")
            folder_ids = [f"{i:02d}" for i in range(1, 51)] # Folders named 01 to 50
        
        print(f"\n{'='*40}")
        print(f"Processing {len(folder_ids)} folders")
        print(f"{'='*40}")
        
        # Check for already processed folders (both .pkl and .pkl.gz)
        processed_folders_dir = []
        processed_folders = []
        for folder_file in list(self.output_dir.glob("folder_*.pkl")) + list(self.output_dir.glob("folder_*.pkl.gz")):
            # Extract folder ID (handle both .pkl and .pkl.gz)
            folder_id = folder_file.stem.replace("folder_", "").replace(".pkl", "")
            processed_folders_dir.append(folder_file)
            processed_folders.append(folder_id)
        
        if processed_folders:
            print(f"\nFound {len(processed_folders)} already processed folders")
            print(f"Will skip: {', '.join(sorted(processed_folders))}")
        
        processed_count = 0
        
        for folder_id in folder_ids:
            # Skip if already processed
            if folder_id in processed_folders:
                print(f"\nSkipping folder {folder_id} (already processed)")
                processed_count += 1
                continue
                
            folder_path = self.data_dir / folder_id
            audio_path = folder_path / f"{folder_id}_phone.wav"
            annotation_path = folder_path / f"{folder_id}_annotation.json"
            
            if (not folder_path.exists() or not audio_path.exists() or not annotation_path.exists()):
                print(f"\nSkipping folder {folder_id}: Missing data or annotation files.")
                continue
            
            print(f"\nProcessing folder {folder_id}...")
            
            try:
                epochs, labels = self.create_epochs(folder_id)
                
                # Save this folder's data immediately
                print(f"\nSaving folder {folder_id} data...")
                self.save_folder_individually(folder_id, epochs, labels)
                
                processed_count += 1
                print(f"Completed folder {folder_id} ({processed_count} folders processed)")
                
            except Exception as e:
                print(f"    Error processing folder {folder_id}: {e}")
                import traceback
                traceback.print_exc()
                continue
            finally:
                # Always clear memory after each folder
                del epochs, labels
                gc.collect()
        
        # After all folders are processed
        if processed_count > 0 or processed_folders:
            print(f"\n{processed_count} folders processed.")
            return processed_folders_dir
        else:
            print("\nNo folders were processed!")
            return None

if __name__ == "__main__":
    preprocessor = AudoPreprocessor(
        data_dir=config.DATA_DIR,
        output_dir=config.PREPROCESSED_DIR,
        epoch_duration=config.AUDIO_EPOCH_DURATION,
        sample_rate=config.AUDIO_SAMPLE_RATE,
        use_compression=config.USE_COMPRESSION,
        extract_features=config.USE_PREEXTRACTED_FEATURES
    )
    
    test_folder_ids = [f"{i:02d}" for i in range(1, config.NUM_AUDIO_FILES+1)] # First 40 Files For Now
    
    data = preprocessor.process_folders(folder_ids=test_folder_ids)

/home/jwethere/.local/lib/python3.12/site-packages/transformers/audio_utils.py:296: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (256) may be set too low.
  warnings.warn(


    Feature extraction will use GPU

Initialized AudoPreprocessor
    Data_dir: /home/jwethere/Data
    Output_dir: /home/jwethere/Preprocessed
    Epoch Duration: 10s
    Sample Rate: 16000Hz
    Samples per epoch: 160000
    Compression: Enabled (gzip)

Processing 50 folders

Found 48 already processed folders
Will skip: 01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50

Skipping folder 01 (already processed)

Skipping folder 02 (already processed)

Skipping folder 03 (already processed)

Skipping folder 04 (already processed)

Skipping folder 05 (already processed)

Skipping folder 06 (already processed)

Skipping folder 07 (already processed)

Skipping folder 08 (already processed)

Skipping folder 09 (already processed)

Skipping folder 10 (already processed)

Skipping folder 11: Missing data or annotation files.

Skipping folder 12 (already 

In [9]:
class MultiEpochSleepApneaDetector(nn.Module):
    # Apnea Detector using Audio Spectrogram Transformer (AST)
    
    # Studies 14->10 Architecture
    # 14 contextual epochs to predict 10 output epochs
    def __init__(self, context_epochs=14, output_epochs=10, num_classes=3, dropout=0.3, lstm_enabled=False, use_preextracted_features=True):
        super().__init__()
        
        self.context_epochs = context_epochs
        self.output_epochs = output_epochs
        self.num_classes = num_classes
        self.lstm_enabled = lstm_enabled
        self.use_preextracted_features = use_preextracted_features
        
        print(f"\nInitializing MultiEpochSleepApneaDetector: ")
        print(f"    Context Epochs: {context_epochs}")
        print(f"    Output Epochs: {output_epochs}")
        print(f"    Number of Classes: {num_classes}")
        print(f"    Dropout: {dropout}")
        print(f"    LSTM Enabled: {self.lstm_enabled}")
        print(f"    Pre-extracted Features: {use_preextracted_features}")

        if not use_preextracted_features:
            # Load pre-trained AST model for on-the-fly extraction
            self.ast_config = ASTConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            self.ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            
            self.ast_model.gradient_checkpointing_enable()
            ast_feature_dim = self.ast_config.hidden_size #768
        else:
            # Features already extracted, just use them
            ast_feature_dim = 768  # AST hidden size
        
        if lstm_enabled:
            # Multi-epoch temporal modeling with LSTM
            self.temporal_lstm = nn.LSTM(
                input_size=ast_feature_dim,
                hidden_size=1024,
                num_layers=3,
                batch_first=True,
                dropout=dropout if context_epochs > 1 else 0,
                bidirectional=True
            )
            
            # Attention layer to focus on relevant time steps
            self.attention = nn.MultiheadAttention(
                embed_dim=2048,  # 1024 * 2 for bidirectional
                num_heads=16,
                dropout=dropout,
                batch_first=True
            )
        
            classifier_input_dim = 2048
        else:
            classifier_input_dim = ast_feature_dim
        
        # Classification layer
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        
        print(f"\nModel initialized successfully.")
        print(f"AST Feature Dimension: {ast_feature_dim}")
        
    def extract_ast_features(self, waveforms):
        # Only used if features are not pre-extracted
        batch_size, n_epochs, n_samples = waveforms.shape
        waveforms_flat = waveforms.reshape(batch_size * n_epochs, n_samples)
        
        inputs = self.ast_feature_extractor(
            [w.cpu().numpy() for w in waveforms_flat], 
            sampling_rate=16000, 
            return_tensors="pt"
        )
        
        inputs = {k: v.to(next(self.parameters()).device) for k, v in inputs.items()}
        
        outputs = self.ast_model(**inputs)
        features = outputs.last_hidden_state[:, 0, :]
            
        features = features.reshape(batch_size, n_epochs, -1)
        return features
    
    def forward(self, input_data):
        # input_data is either raw waveforms or pre-extracted features
        if self.use_preextracted_features:
            # Input is already features: (batch, epochs, 768)
            features = input_data
        else:
            # Extract features from raw audio
            features = self.extract_ast_features(input_data)

        if self.lstm_enabled:
            lstm_out, _ = self.temporal_lstm(features)
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
            
            start_idx = (self.context_epochs - self.output_epochs) // 2
            end_idx = start_idx + self.output_epochs
            output_features = attn_out[:, start_idx:end_idx, :]
        else:
            start_idx = (self.context_epochs - self.output_epochs) // 2
            end_idx = start_idx + self.output_epochs
            output_features = features[:, start_idx:end_idx, :]
        
        logits = self.classifier(output_features)
        return logits
    
class WeightedCrossEntropyLoss(nn.Module):
    # Weighted Cross Entropy Loss for class imbalance
    # Study --> 1.0 for no, 1.3 for apnea, 2.1 for hypopnea
    
    def __init__(self, weights=None):
        super().__init__()
        
        if weights is None:
            weights = torch.tensor([1.0, 1.3, 2.1])
            
        self.weights = weights
        print(f"\nInitialized WeightedCrossEntropyLoss with weights: {self.weights}")
        
    def forward(self, logits, targets):
        weights = self.weights.to(logits.device)
        
        # Reshape
        batch_size, n_epochs, n_classes = logits.shape
        logits_flat = logits.reshape(-1, n_classes)
        targets_flat = targets.reshape(-1)
        
        # Calculate loss
        loss = F.cross_entropy(logits_flat, targets_flat, weight=weights)
        
        return loss
class SleepApneaDataset(Dataset):
    def __init__(self, preprocessed_dir, context_epochs=14, output_epochs=10, use_compression=True, cache_size=3, index_path=None):
        self.preprocessed_dir = Path(preprocessed_dir)
        self.context_epochs = context_epochs
        self.output_epochs = output_epochs
        self.use_compression = use_compression
        self.cache_size = cache_size
        
        self.folder_cache = {}
        self.cache_order = []
        
        if index_path is None:
            index_path = self.preprocessed_dir / "index.json"
        else:
            index_path = Path(index_path)
            
        self.index_path = index_path
        
        if index_path.exists():
            print(f"\nLoading dataset index from {index_path}...")
            self._load_index(index_path)
        else:
            print(f"\nNo index found at {index_path}. Building index...")
            self.folder_files, self.folder_metadata, self.valid_indices = self._build_index()
            self._save_index(index_path)
        
        print(f"\nInitialized SleepApneaDataset:")
        print(f"    Total folders: {len(self.folder_files)}")
        print(f"    Valid sequences: {len(self.valid_indices)}")
        print(f"    Context epochs: {self.context_epochs}")
        print(f"    Output epochs: {self.output_epochs}")
        print(f"    Cache size: {self.cache_size} folders")
    
    def _save_index(self, path):
        index = {
            "folder_metadata": {
                k: {
                    "file_path": v["file_path"].name,
                    "num_epochs": v["num_epochs"],
                    "start_idx": v["start_idx"],
                    "end_idx": v["end_idx"],
                }
                for k, v in self.folder_metadata.items()
            },
            "valid_indices": self.valid_indices,
        }
        
        with open(path, "w") as f:
            json.dump(index, f)
            
        print(f"\nIndex saved to {path}")
        
    def _load_index(self, path):
        with open(path, "r") as f:
            index = json.load(f)

        self.folder_metadata = {
            k: {
                **v,
                "file_path": self.preprocessed_dir / Path(v["file_path"])
            }
            for k, v in index["folder_metadata"].items()
        }

        self.valid_indices = index["valid_indices"]
        self.folder_files = [v["file_path"] for v in self.folder_metadata.values()]
    
    def _build_index(self):
        file_ext = ".pkl.gz" if self.use_compression else ".pkl"
        folder_files = sorted(self.preprocessed_dir.glob(f"folder_*{file_ext}"))
        
        if not folder_files:
            raise FileNotFoundError(f"No preprocessed files found in {self.preprocessed_dir}")
        
        print(f"\nIndexing {len(folder_files)} preprocessed files...")
        
        folder_metadata = {}
        global_epoch_idx = 0
        
        # Build metadata for each folder
        for folder_file in folder_files:
            # Load only metadata, not the actual data
            if self.use_compression:
                with gzip.open(folder_file, 'rb') as f:
                    data = pickle.load(f)
            else:
                with open(folder_file, 'rb') as f:
                    data = pickle.load(f)
            
            folder_id = data['folder_id']
            num_epochs = len(data['labels'])
            
            folder_metadata[folder_id] = {
                'file_path': folder_file,
                'num_epochs': num_epochs,
                'start_idx': global_epoch_idx,
                'end_idx': global_epoch_idx + num_epochs
            }
            
            global_epoch_idx += num_epochs
            print(f"    {folder_file.name}: {num_epochs} epochs")
        
        # Build valid sequence indices
        valid_indices = []
        
        for folder_id, metadata in folder_metadata.items():
            folder_num_epochs = metadata['num_epochs']
            folder_start = metadata['start_idx']
            
            # Create sequences within this folder only
            num_sequences = folder_num_epochs - self.context_epochs + 1
            
            for i in range(num_sequences):
                global_idx = folder_start + i
                valid_indices.append({
                    'global_idx': global_idx,
                    'folder_id': folder_id,
                    'local_idx': i  # Index within the folder
                })
        
        print(f"\nTotal valid sequences: {len(valid_indices)}")
        
        return folder_files, folder_metadata, valid_indices
    
    def _load_folder(self, folder_id):
        if folder_id in self.folder_cache:
            return self.folder_cache[folder_id]
        
        # Load the folder data
        file_path = self.folder_metadata[folder_id]['file_path']
        
        if self.use_compression:
            with gzip.open(file_path, 'rb') as f:
                data = pickle.load(f)
        else:
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
        
        # Cache management (LRU-style)
        if len(self.folder_cache) >= self.cache_size:
            # Remove oldest folder from cache
            oldest_folder = self.cache_order.pop(0)
            del self.folder_cache[oldest_folder]
        
        # Add to cache
        self.folder_cache[folder_id] = {
            'epochs': data['epochs'],
            'labels': data['labels']
        }
        self.cache_order.append(folder_id)
        
        return self.folder_cache[folder_id]

    def _get_folder_split_indices(self, train_folders, val_folders):
        train_indices = []
        val_indices = []

        for idx, seq_info in enumerate(self.valid_indices):
            folder_id = seq_info['folder_id']
            if folder_id in train_folders:
                train_indices.append(idx)
            elif folder_id in val_folders:
                val_indices.append(idx)

        return train_indices, val_indices
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        sequence_info = self.valid_indices[idx]
        folder_id = sequence_info['folder_id']
        local_idx = sequence_info['local_idx']
        
        # Load folder data (from cache or disk)
        folder_data = self._load_folder(folder_id)
        
        # Extract sequence from this folder
        start_idx = local_idx
        context = folder_data['epochs'][start_idx:start_idx + self.context_epochs]
        
        label_start = start_idx + (self.context_epochs - self.output_epochs) // 2
        label_end = label_start + self.output_epochs
        labels = folder_data['labels'][label_start:label_end]
        
        if isinstance(context, np.ndarray):
            context_tensor = torch.from_numpy(context).float()
        else:
            context_tensor = torch.FloatTensor(context)
        
        if isinstance(labels, np.ndarray):
            labels_tensor = torch.from_numpy(labels).long()
        else:
            labels_tensor = torch.LongTensor(labels)
        
        return context_tensor, labels_tensor
    
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, device, output_dir, scheduler = None, patience=10, gradient_accumulation_steps=1, use_mixed_precision=True):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.output_dir = Path(output_dir)
        self.patience = patience
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.use_mixed_precision = use_mixed_precision
        
        # Initialize gradient scaler for mixed precision
        self.scaler = GradScaler(device=self.device.type) if use_mixed_precision else None
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # History
        self.history = {
            'train_loss': [],
            'train_accuracy': [],
            'val_loss': [],
            'val_accuracy': [],
            'val_f1': []
        }
        
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        self.epochs_no_improve = 0
        
        # Stats Monitoring
        if torch.cuda.is_available():
            try:
                pynvml.nvmlInit()
                self.nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
                self.use_nvml = True
            except:
                self.use_nvml = False
                print("Warning: Could not initialize NVML for detailed GPU stats")
        else:
            self.use_nvml = False
        
        print(f"\nInitialized Trainer:")
        print(f"    Device: {self.device}")
        print(f"    Output Directory: {self.output_dir}")
        print(f"    Training Batches: {len(self.train_loader)}")
        print(f"    Val Batches: {len(self.val_loader)}")
        print(f"    Patience: {self.patience} epochs")
        print(f"    Mixed Precision: {self.use_mixed_precision}")
        print(f"    Gradient Accumulation: {self.gradient_accumulation_steps}")
    
    def get_gpu_stats(self):
        if not torch.cuda.is_available():
            return {}
        
        stats = {}
        
        # Memory stats
        stats['mem_alloc'] = torch.cuda.memory_allocated() / 1024**3
        stats['mem_reserved'] = torch.cuda.memory_reserved() / 1024**3
        
        # Detailed stats with NVML
        if self.use_nvml:
            try:
                mem_info = pynvml.nvmlDeviceGetMemoryInfo(self.nvml_handle)
                stats['mem_used'] = mem_info.used / 1024**3
                stats['mem_total'] = mem_info.total / 1024**3
                
                util = pynvml.nvmlDeviceGetUtilizationRates(self.nvml_handle)
                stats['gpu_util'] = util.gpu
                
                temp = pynvml.nvmlDeviceGetTemperature(self.nvml_handle, pynvml.NVML_TEMPERATURE_GPU)
                stats['temp'] = temp
            except:
                pass
        
        return stats
    
    def train_epoch(self, epoch):
        self.model.train()
        
        total_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch} [Training]", leave=False)
        self.optimizer.zero_grad()
        
        for batch_idx, (data, targets) in enumerate(pbar):
            data, targets = data.to(self.device), targets.to(self.device)
            
            # Mixed precision training
            if self.use_mixed_precision:
                with autocast(device_type=self.device.type):
                    logits = self.model(data)
                    loss = self.criterion(logits, targets) / self.gradient_accumulation_steps
                
                self.scaler.scale(loss).backward()
            else:
                logits = self.model(data)
                loss = self.criterion(logits, targets) / self.gradient_accumulation_steps
                loss.backward()
            
            # Update weights every N steps
            if (batch_idx + 1) % self.gradient_accumulation_steps == 0:
                if self.use_mixed_precision:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()
                
                self.optimizer.zero_grad()
            
            # Statistics
            total_loss += loss.item() * self.gradient_accumulation_steps
            predictions = logits.argmax(dim=-1)
            correct += (predictions == targets).sum().item()
            total += targets.numel()
            
            # Update Dict
            postfix_dict = {
                'Loss': f"{total_loss/(batch_idx+1):.4f}",
                'Acc': f"{100.0 * correct / total:.2f}%"
            }
            
            # Add GPU/Memory stats
            gpu_stats = self.get_gpu_stats()
            if gpu_stats:
                postfix_dict['GPU_Mem'] = f"{gpu_stats['mem_alloc']:.1f}GB"
                if 'gpu_util' in gpu_stats:
                    postfix_dict['GPU%'] = f"{gpu_stats['gpu_util']}%"
                if 'temp' in gpu_stats:
                    postfix_dict['Temp'] = f"{gpu_stats['temp']}°C"
            
            # System RAM
            postfix_dict['RAM'] = f"{psutil.virtual_memory().percent:.1f}%"
            
            pbar.set_postfix(postfix_dict)
            
            # Clear cache periodically
            if batch_idx % 100 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Handle remaining gradients
        if (batch_idx + 1) % self.gradient_accumulation_steps != 0:
            if self.use_mixed_precision:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            
            self.optimizer.zero_grad()
        
        avg_loss = total_loss / len(self.train_loader)
        accuracy = 100.0 * correct / total
        
        return avg_loss, accuracy
    
    def validate_epoch(self, epoch):
        self.model.eval()
        
        total_loss = 0.0
        correct = 0
        total = 0

        # Per-class tracking
        class_correct = {0: 0, 1: 0, 2: 0}  # no_event, osa, hypo
        class_total = {0: 0, 1: 0, 2: 0}
        
        all_predictions = []
        all_targets = []
        
        pbar = tqdm(self.val_loader, desc=f"Epoch {epoch} [Validation]", leave=False)
        
        with torch.no_grad():
            for data, targets in pbar:
                data, targets = data.to(self.device), targets.to(self.device)
                
                # Mixed precision for validation too
                if self.use_mixed_precision:
                    with autocast(device_type=self.device.type):
                        logits = self.model(data)
                        loss = self.criterion(logits, targets)
                else:
                    logits = self.model(data)
                    loss = self.criterion(logits, targets)
                
                # Statistics
                total_loss += loss.item()
                predictions = logits.argmax(dim=-1)
                correct += (predictions == targets).sum().item()
                total += targets.numel()

                # Per-class statistics
                for class_idx in [0, 1, 2]:
                    class_mask = (targets == class_idx)
                    class_total[class_idx] += class_mask.sum().item()
                    class_correct[class_idx] += ((predictions == targets) & class_mask).sum().item()
                
                # Update Dict
                postfix_dict = {
                    'Loss': f"{total_loss/(pbar.n+1):.4f}",
                    'Acc': f"{100.0 * correct / total:.2f}%"
                }
                
                # Add GPU/Memory stats
                gpu_stats = self.get_gpu_stats()
                if gpu_stats:
                    postfix_dict['GPU_Mem'] = f"{gpu_stats['mem_alloc']:.1f}GB"
                    if 'gpu_util' in gpu_stats:
                        postfix_dict['GPU%'] = f"{gpu_stats['gpu_util']}%"
                    if 'temp' in gpu_stats:
                        postfix_dict['Temp'] = f"{gpu_stats['temp']}°C"
                
                # System RAM
                postfix_dict['RAM'] = f"{psutil.virtual_memory().percent:.1f}%"

                # Progres Update
                pbar.set_postfix(postfix_dict)
                
                all_predictions.extend(predictions.cpu().numpy().flatten())
                all_targets.extend(targets.cpu().numpy().flatten())
        
        avg_loss = total_loss / len(self.val_loader)
        accuracy = 100.0 * correct / total

        # Calculate per-class accuracy
        class_names = {0: 'No Event', 1: 'OSA', 2: 'Hypopnea'}
        per_class_acc = {}
        for class_idx in [0, 1, 2]:
            if class_total[class_idx] > 0:
                per_class_acc[class_idx] = 100.0 * class_correct[class_idx] / class_total[class_idx]
            else:
                per_class_acc[class_idx] = 0.0
        
        # Calculate F1 Scores
        f1_macro = f1_score(all_targets, all_predictions, average='macro', zero_division=0)
        f1_per_class = f1_score(all_targets, all_predictions, average=None, zero_division=0)

        # Metrics
        metrics = {
            'loss': avg_loss,
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'class_metrics': {
                class_names[i]: {
                    'accuracy': per_class_acc[i],
                    'f1': f1_per_class[i],
                    'count': class_total[i]
                }
                for i in [0, 1, 2]
            }
        }
        
        return metrics, all_predictions, all_targets
    
    def train(self, num_epochs):
        print("\n" + "="*40)
        print(f"Starting training for {num_epochs} epochs...")
        print("="*40)
        
        for epoch in range(1, num_epochs + 1):
            print(f"\nEpoch {epoch}/{num_epochs}")
            print("-"*40)
            
            train_loss, train_acc = self.train_epoch(epoch)
            val_metrics, _, _ = self.validate_epoch(epoch)

            if self.scheduler is not None:
                self.scheduler.step(val_metrics['loss'])
            
            # Log history
            self.history['train_loss'].append(train_loss)
            self.history['train_accuracy'].append(train_acc)
            self.history['val_loss'].append(val_metrics['loss'])
            self.history['val_accuracy'].append(val_metrics['accuracy'])
            self.history['val_f1'].append(val_metrics['f1_macro'])

            # Class history
            if 'per_class_accuracy' not in self.history:
                self.history['per_class_accuracy'] = {
                    'No Event': [],
                    'OSA': [],
                    'Hypopnea': []
                }
                self.history['per_class_f1'] = {
                    'No Event': [],
                    'OSA': [],
                    'Hypopnea': []
                }

            for class_name, metrics in val_metrics['class_metrics'].items():
                self.history['per_class_accuracy'][class_name].append(metrics['accuracy'])
                self.history['per_class_f1'][class_name].append(metrics['f1'])

            # Print Summary
            print(f"\nEpoch {epoch} Summary:")
            print(f"    Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"    Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.2f}%, Val F1: {val_metrics['f1_macro']:.4f}")
            
            print(f"\n    Per-Class Validation Metrics:")
            print(f"    {'Class':<15} {'Count':<8} {'Accuracy':<12} {'F1 Score':<10}")
            print(f"    {'-'*50}")
            for class_name, metrics in val_metrics['class_metrics'].items():
                print(f"    {class_name:<15} {metrics['count']:<8} {metrics['accuracy']:>6.2f}%      {metrics['f1']:>6.4f}")
        
            
            # Check for improvement
            if val_metrics['loss'] < self.best_val_loss:
                self.best_val_loss = val_metrics['loss']
                self.best_epoch = epoch
                self.epochs_no_improve = 0
                
                checkpoint_path = self.output_dir / "best_model.pth"
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_loss': val_metrics['loss'],
                    'val_accuracy': val_metrics['accuracy'],
                    'val_f1': val_metrics['f1_macro'],
                    'per_class_metrics': val_metrics['class_metrics']
                }, checkpoint_path)
                
                print(f"    New best model saved (Val Loss: {val_metrics['loss']:.4f})")
            else:
                self.epochs_no_improve += 1
                print(f"    No improvement for {self.epochs_no_improve} epochs.")
            
            # Early stopping
            if self.epochs_no_improve >= self.patience:
                print(f"\nEarly stopping triggered after {self.patience} epochs with no improvement.")
                print(f"Best model from epoch {self.best_epoch} with Val Loss: {self.best_val_loss:.4f}")
                break
            
            # Periodic checkpoints
            if epoch % 5 == 0:
                checkpoint_path = self.output_dir / f'checkpoint_epoch_{epoch}.pth'
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'history': self.history
                }, checkpoint_path)
                print(f"    Checkpoint saved to checkpoint_epoch_{epoch}.pth")
        
        # Save training history
        history_path = self.output_dir / "training_history.json"
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f"\nTraining history saved to {history_path}")
        
        print(f"\n{'='*40}")
        print(f"Training Complete!")
        print(f"   Total epochs: {epoch}")
        print(f"   Best epoch: {self.best_epoch}")
        print(f"   Best Val Loss: {self.best_val_loss:.4f}")
        print(f"{'='*40}")
        
if __name__ == "__main__":
    print("\n" + "="*60)
    print("Sleep Apnea Detection Metadata Prep")
    print("="*60)
    
    # Create Dataset
    print(f"\nLoading preprocessed data from {config.PREPROCESSED_DIR}...")
    dataset = SleepApneaDataset(
        preprocessed_dir=config.PREPROCESSED_DIR,
        context_epochs=config.CONTEXT_EPOCHS,
        output_epochs=config.OUTPUT_EPOCHS,
        use_compression=config.USE_COMPRESSION,
        cache_size=config.CACHE_SIZE
    )


Sleep Apnea Detection Metadata Prep

Loading preprocessed data from /home/jwethere/Preprocessed...

Loading dataset index from /home/jwethere/Preprocessed/index.json...

Initialized SleepApneaDataset:
    Total folders: 48
    Valid sequences: 138319
    Context epochs: 14
    Output epochs: 10
    Cache size: 48 folders


In [10]:
# Primary Code

if __name__ == "__main__":
    # TF32
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("\n" + "="*60)
    print("Sleep Apnea Detection Training")
    print("="*60)
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version: {torch.version.cuda}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    print(f"Gradient Accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"Effective Batch Size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"Num Workers: {config.NUM_WORKERS}")
    print(f"Mixed Precision: {config.USE_MIXED_PRECISION}")
    print(f"Cache Size: {config.CACHE_SIZE}")
    print("="*60)
    
    # Create Dataset (now loads from multiple files automatically)
    print(f"\nLoading preprocessed data from {config.PREPROCESSED_DIR}...")
    dataset = SleepApneaDataset(
        preprocessed_dir=config.PREPROCESSED_DIR,
        context_epochs=config.CONTEXT_EPOCHS,
        output_epochs=config.OUTPUT_EPOCHS,
        use_compression=config.USE_COMPRESSION,
        cache_size=config.CACHE_SIZE,
        index_path=config.PREPROCESSED_DIR+"/index.json"
    )

    # Train/Val Split By Folder (Splitting by patient to avoid overlaps)
    all_folder_ids = list(dataset.folder_metadata.keys())
    np.random.seed(42) #Set seed for testing
    np.random.shuffle(all_folder_ids)

    split_idx = int(len(all_folder_ids) * (1 - config.VAL_SPLIT))
    train_folder_ids = all_folder_ids[:split_idx]
    val_folder_ids = all_folder_ids[split_idx:]

    print(f"\nFolder-level split:")
    print(f"    Total folders: {len(all_folder_ids)}")
    print(f"    Training folders: {len(train_folder_ids)} ({train_folder_ids[:5]}...)")
    print(f"    Validation folders: {len(val_folder_ids)} ({val_folder_ids[:5]}...)")

    train_indices, val_indices = dataset._get_folder_split_indices(train_folder_ids, val_folder_ids)

    print(f"\nDataset split:")
    print(f"    Training samples: {len(train_indices)}")
    print(f"    Validation samples: {len(val_indices)}")

    # Dataset Subsets
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.BATCH_SIZE, 
        shuffle=True, 
        num_workers=config.NUM_WORKERS,
        pin_memory=True if device.type == 'cuda' else False,
        persistent_workers=(config.NUM_WORKERS > 0),
        prefetch_factor=8 if config.NUM_WORKERS > 0 else None
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True if device.type == 'cuda' else False,
        persistent_workers=(config.NUM_WORKERS > 0),
        prefetch_factor=8 if config.NUM_WORKERS > 0 else None
    )
    
    # Initialize model
    print("\n" + "="*40)
    print("Initializing model...")
    print("="*40)
    
    model = MultiEpochSleepApneaDetector(
        context_epochs=config.CONTEXT_EPOCHS,
        output_epochs=config.OUTPUT_EPOCHS,
        num_classes=config.NUM_CLASSES,
        dropout=config.DROPOUT,
        lstm_enabled=config.LSTM_ENABLED,
        use_preextracted_features=config.USE_PREEXTRACTED_FEATURES
    ).to(device)
    
    # Loss and Optimizer
    criterion = WeightedCrossEntropyLoss(
        weights=torch.tensor(config.CLASS_WEIGHTS)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=0.01,
        fused=True if device.type == 'cuda' else False
    )
    
    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        output_dir=config.OUTPUT_DIR,
        scheduler=scheduler,
        patience=config.PATIENCE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
        use_mixed_precision=config.USE_MIXED_PRECISION
    )
    
    # Start training
    trainer.train(num_epochs=config.NUM_EPOCHS)
    
    print("\nTraining script complete!")


Sleep Apnea Detection Training
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
CUDA Version: 12.4
Batch Size: 128
Gradient Accumulation: 1
Effective Batch Size: 128
Num Workers: 8
Mixed Precision: True
Cache Size: 48

Loading preprocessed data from /home/jwethere/Preprocessed...

Loading dataset index from /home/jwethere/Preprocessed/index.json...

Initialized SleepApneaDataset:
    Total folders: 48
    Valid sequences: 138319
    Context epochs: 14
    Output epochs: 10
    Cache size: 48 folders

Folder-level split:
    Total folders: 48
    Training folders: 38 (['30', '43', '29', '46', '27']...)
    Validation folders: 10 (['12', '25', '21', '50', '23']...)

Dataset split:
    Training samples: 108820
    Validation samples: 29499

Initializing model...

Initializing MultiEpochSleepApneaDetector: 
    Context Epochs: 14
    Output Epochs: 10
    Number of Classes: 3
    Dropout: 0.3
    LSTM Enabled: True
    Pre-extracted Features: True

Model initialized successfully.
AST Feature Dimen


Epoch 1 Summary:
    Train Loss: 0.6302, Train Acc: 72.07%
    Val Loss: 0.7577, Val Acc: 69.21%, Val F1: 0.5339

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    77.24%      0.8164
    OSA             28090     63.51%      0.4739
    Hypopnea        44648     32.81%      0.3113
    New best model saved (Val Loss: 0.7577)

Epoch 2/50
----------------------------------------



Epoch 2 Summary:
    Train Loss: 0.4924, Train Acc: 78.75%
    Val Loss: 0.9360, Val Acc: 71.71%, Val F1: 0.5339

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    82.36%      0.8362
    OSA             28090     53.65%      0.4510
    Hypopnea        44648     30.05%      0.3144
    No improvement for 1 epochs.

Epoch 3/50
----------------------------------------



Epoch 3 Summary:
    Train Loss: 0.3716, Train Acc: 84.86%
    Val Loss: 1.3146, Val Acc: 72.09%, Val F1: 0.5399

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    82.72%      0.8421
    OSA             28090     48.90%      0.4521
    Hypopnea        44648     33.76%      0.3255
    No improvement for 2 epochs.

Epoch 4/50
----------------------------------------



Epoch 4 Summary:
    Train Loss: 0.2587, Train Acc: 89.90%
    Val Loss: 1.6423, Val Acc: 73.34%, Val F1: 0.5374

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    85.44%      0.8488
    OSA             28090     52.51%      0.4695
    Hypopnea        44648     26.23%      0.2939
    No improvement for 3 epochs.

Epoch 5/50
----------------------------------------


...


Epoch 7 Summary:
    Train Loss: 0.0928, Train Acc: 96.85%
    Val Loss: 2.5087, Val Acc: 74.05%, Val F1: 0.5324

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    86.96%      0.8546
    OSA             28090     55.21%      0.4762
    Hypopnea        44648     21.65%      0.2666
    No improvement for 6 epochs.

Epoch 8/50
----------------------------------------



Epoch 8 Summary:
    Train Loss: 0.0510, Train Acc: 98.30%
    Val Loss: 2.8918, Val Acc: 73.68%, Val F1: 0.5251

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    87.11%      0.8533
    OSA             28090     47.02%      0.4510
    Hypopnea        44648     23.56%      0.2710
    No improvement for 7 epochs.

Epoch 9/50
----------------------------------------



Epoch 9 Summary:
    Train Loss: 0.0406, Train Acc: 98.66%
    Val Loss: 2.8919, Val Acc: 73.42%, Val F1: 0.5217

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    86.86%      0.8522
    OSA             28090     47.49%      0.4505
    Hypopnea        44648     22.82%      0.2624
    No improvement for 8 epochs.

Epoch 10/50
----------------------------------------



Epoch 10 Summary:
    Train Loss: 0.0346, Train Acc: 98.85%
    Val Loss: 3.2045, Val Acc: 73.79%, Val F1: 0.5220

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    87.57%      0.8552
    OSA             28090     44.20%      0.4397
    Hypopnea        44648     23.77%      0.2710
    No improvement for 9 epochs.
    Checkpoint saved to checkpoint_epoch_10.pth

Epoch 11/50
----------------------------------------



Epoch 11 Summary:
    Train Loss: 0.0297, Train Acc: 99.00%
    Val Loss: 3.3550, Val Acc: 73.44%, Val F1: 0.5206

    Per-Class Validation Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        222252    86.93%      0.8522
    OSA             28090     47.98%      0.4493
    Hypopnea        44648     22.31%      0.2602
    No improvement for 10 epochs.

Early stopping triggered after 10 epochs with no improvement.
Best model from epoch 1 with Val Loss: 0.7577

Training history saved to /home/jwethere/Model_Output/training_history.json

Training Complete!
   Total epochs: 11
   Best epoch: 1
   Best Val Loss: 0.7577

Training script complete!
